# 14 — Agentes na origem, hub e fonte entre contas

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 4 contas AWS distintas

**Esta aula não executa em uma conta única.** O fluxo mostrado envolve 3 contas (retail-banking, platform, transaction-banking), mas o manifesto e o IaC atuais exigem as quatro contas preparadas: plataforma, Retail, Transaction e Lending. Sem elas, reserve a aula para demonstração do instrutor no ambiente completo ou retire-a da seleção de execução.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 14](../docs/diagramas/svg/14_cenario_santander_integrado.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "14")
print({"lab": "14", "dependencies": list(dependencies), "region": config["region"]})

### Carregar o catálogo implantado

O catálogo tem roles diferentes para desenvolver, publicar, aprovar e consumir. Os SDKs atuais são agent-registry-control para governança e agent-registry para descoberta.

In [ ]:
from santander_demo.registry_workflow.client import RegistryLab
registry_config = json.loads((ROOT / dependencies["registry"]["config"]).read_text())
registry = RegistryLab(registry_config)
assert registry_config["RegistryArn"].startswith("arn:aws:agent-registry:")
print({"registry": registry_config["RegistryArn"], "states": registry.states()})

### Localizar as capacidades publicadas

O Registry oferece metadados aprovados. Endpoints de execução permanecem limitados aos outputs do IaC, impedindo que uma descrição de catálogo redirecione credenciais. Os registros Riscos e Fraude descrevem os dois Runtimes desta aula.

In [ ]:
from santander_demo.aws_workshop.catalog import origin_bundle, validate_origin_record
origin = AWSWorkshop(CONFIG, "retail-banking")
runtimes = {label: origin.outputs("agent-" + label.lower())["RuntimeArn"]
            for label in ["Riscos", "Fraude"]}
expected = origin_bundle(origin.identity,
                         AWSWorkshop(CONFIG).gateway["GatewayUrl"], runtimes)
records = registry.discover()
discovered = {}
reviewer = registry.persona("Approver").client("agent-registry-control")
for label in ["Riscos", "Fraude"]:
    record = next(x for x in records if x["recordId"] == registry_config[label + "RecordId"])
    assert validate_origin_record(record, expected[label]) == runtimes[label]
    state = reviewer.get_registry_record(
        registryId=registry_config["RegistryId"], recordId=record["recordId"])
    assert state["status"] == "APPROVED"
    discovered[label] = record
    display(Code(record["descriptors"]["custom"]["data"], language="json"))
print({label: value["recordId"] for label, value in discovered.items()})

### Examinar as fases de governança dessas capacidades

O pipeline workshop_catalog.py publica descriptors imutáveis, submete cada registro e aplica a decisão com seu digest revisado. Os registros dos fundamentos permanecem em seus próprios stacks.

In [ ]:
for phase in ["origin-publish", "origin-submit", "origin-approve"]:
    display(Code((ROOT / "infra/registry_workflow/templates" / (phase + ".json")).read_text(),
                 language="json"))
display(Code(inspect.getsource(validate_origin_record), language="python"))

### Executar os agentes na origem A e consultar a fonte B via hub C

Riscos e Fraude são dois Runtimes na conta Retail. Eles usam o Gateway da plataforma para chegar ao Gateway de Transaction. O Runtime da fonte verifica pessoa/aplicação e consulta a ACL.

In [ ]:
risk_token = origin.login("c", "riscos")
fraud_token = origin.login("c", "supervisor")
risk = await origin.invoke_agent(risk_token, "Consulte o risco de C001.",
                                 domain="transaction-banking", phase="agent-riscos")
fraud = await origin.invoke_agent(fraud_token, "Consulte os indícios de fraude de C001.",
                                  domain="transaction-banking", phase="agent-fraude")
assert all(x["execution"] == "AWS_AGENTCORE_RUNTIME" for x in [risk, fraud])
print(json.dumps({
    "riscos": {"registry_record_id": discovered["Riscos"]["recordId"], **risk},
    "fraude": {"registry_record_id": discovered["Fraude"]["recordId"], **fraud},
}, indent=2, ensure_ascii=False))

### Negar a audiência da outra aplicação na entrada do Runtime

Um registro aprovado não altera o authorizer. O token de Supervisor não deve entrar no Runtime de Riscos, nem o token de Riscos no de Fraude. A rejeição deve acontecer antes do agente.

In [ ]:
import httpx
from urllib.parse import quote
denied = []
async with httpx.AsyncClient(timeout=60, follow_redirects=False) as http:
    for label, wrong_token in [("Riscos", fraud_token), ("Fraude", risk_token)]:
        endpoint = (f"https://bedrock-agentcore.{config['region']}.amazonaws.com/runtimes/"
                    + quote(runtimes[label], safe="") + "/invocations?qualifier=DEFAULT")
        response = await http.post(endpoint, headers={
            "Authorization": "Bearer " + wrong_token,
            "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4()),
        }, json={"question": "Consulte C001.", "domain": "transaction-banking"})
        assert response.status_code in {401, 403}
        assert response.headers.get("x-amzn-requestid")
        denied.append({"agent": label, "wrong_audience": "DENIED",
                       "runtime_request_id": response.headers["x-amzn-requestid"]})
print(denied)

**Limite da demonstração:** os serviços AgentCore e as contas são reais. O backend Databricks é um simulador AWS explicitamente identificado. A aprovação no Registry, a decisão Cedar e o grant na fonte são evidências distintas.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.